# AIO Appearance — Test Collection
Sanity-checks Google AI Overview appearance for the test batch collected from `queries/human_short_queries.csv`.
Loads all `serp_raw_*.json` files from `data/raw_test` (kept separate from the main `data/raw` dataset).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from common import (
    load_data,
    set_plot_style,
    PALETTE,
    ROOT,
)

sns.set_theme(style="whitegrid")
set_plot_style()

RAW_TEST_DIR = ROOT / "data" / "raw_test"
df = load_data(raw_dir=RAW_TEST_DIR)

## 0. Query counts

In [ ]:
n_records = len(df)
n_unique_queries = df["query"].nunique()

print(f"Total records (SERP requests) : {n_records}")
print(f"Unique queries                : {n_unique_queries}")

print("\nBy stance:")
print(df["stance"].value_counts().reindex(["Pro", "Neutral", "Con"]).to_string())

print("\nBy leaning:")
print(df["pro_leaning"].value_counts().reindex(["Left", "Right"]).to_string())

## 1. AIO presence per topic

In [ ]:
topic_stats = (
    df.groupby("topic")["has_ai_overview"]
    .agg(total="count", aio_present="sum")
    .assign(
        aio_absent=lambda x: x["total"] - x["aio_present"],
        aio_rate=lambda x: x["aio_present"] / x["total"],
    )
    .sort_values("aio_rate", ascending=False)
)
display(topic_stats.style.format({"aio_rate": "{:.0%}"}))

fig, ax = plt.subplots(figsize=(10, 4))
x = range(len(topic_stats))
ax.bar(x, topic_stats["aio_present"], label="AIO present", color="steelblue")
ax.bar(
    x,
    topic_stats["aio_absent"],
    bottom=topic_stats["aio_present"],
    label="AIO absent",
    color="#d9534f",
)
ax.set_xticks(list(x))
ax.set_xticklabels(topic_stats.index, rotation=30, ha="right")
ax.set_ylabel("Number of requests")
ax.set_title("AIO presence per topic (test batch)")
ax.legend()
plt.tight_layout()
plt.show()

## 1b. AIO fetch attempts — first try, retry, or never
`aio_fetch_attempts` records how many SerpAPI requests were made before AIO appeared (or the collector gave up).

In [ ]:
def _attempt_bucket(row):
    if row["aio_fetch_attempts"] <= 1:
        return "First attempt"
    elif row["has_ai_overview"]:
        return "Retry succeeded"
    else:
        return "Never (gave up)"


df["aio_attempt_bucket"] = df.apply(_attempt_bucket, axis=1)
ATTEMPT_ORDER = ["First attempt", "Retry succeeded", "Never (gave up)"]

attempt_counts = (
    df["aio_attempt_bucket"].value_counts().reindex(ATTEMPT_ORDER).fillna(0).astype(int)
)
print(attempt_counts.to_string())
print(f"\nShare of all queries: \n{(attempt_counts / len(df)).round(3).to_string()}")

fig, ax = plt.subplots(figsize=(6, 4))
colors = ["#2ecc71", "#f39c12", "#e74c3c"]
bars = ax.bar(ATTEMPT_ORDER, attempt_counts.values, color=colors)
ax.bar_label(bars, padding=3)
ax.set_ylabel("Number of queries")
ax.set_title("AIO fetch outcome — first attempt vs retry vs never (test batch)")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()

## 2. AIO presence per stance (Pro / Neutral / Con)

In [ ]:
df_stance = df[df["stance"].notna()]

if df_stance.empty:
    print("No stance data yet.")
else:
    stance_stats = (
        df_stance.groupby("stance")["has_ai_overview"]
        .agg(total="count", aio_present="sum")
        .assign(aio_rate=lambda x: x["aio_present"] / x["total"])
        .reindex(["Pro", "Neutral", "Con"])
    )
    display(stance_stats.style.format({"aio_rate": "{:.0%}"}))

    fig, ax = plt.subplots(figsize=(6, 4))
    for i, (stance, row) in enumerate(stance_stats.iterrows()):
        ax.bar(i, row["aio_rate"], color=PALETTE.get(stance, "grey"), label=stance)
    ax.set_xticks(range(len(stance_stats)))
    ax.set_xticklabels(stance_stats.index)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    ax.set_ylabel("AIO presence rate")
    ax.set_title("AIO rate by query stance (test batch)")
    ax.set_ylim(0, 1)
    plt.tight_layout()
    plt.show()

## 2b. AIO presence by political leaning × stance

In [ ]:
df_lean = df[df["pro_leaning"].notna() & df["stance"].notna()].copy()

if df_lean.empty:
    print("No pro_leaning / stance data yet.")
else:
    LEANING_ORDER = ["Left", "Right"]
    STANCE_ORDER = ["Pro", "Neutral", "Con"]

    pivot = (
        df_lean.groupby(["pro_leaning", "stance"])["has_ai_overview"]
        .agg(total="count", aio_present="sum")
        .assign(aio_rate=lambda x: x["aio_present"] / x["total"])
        .reindex(
            pd.MultiIndex.from_product(
                [LEANING_ORDER, STANCE_ORDER], names=["pro_leaning", "stance"]
            )
        )
    )
    display(pivot.style.format({"aio_rate": "{:.0%}"}))

    heat = (
        pivot["aio_rate"]
        .unstack("stance")
        .reindex(index=LEANING_ORDER, columns=STANCE_ORDER)
    )
    fig, ax = plt.subplots(figsize=(5, 2.5))
    sns.heatmap(
        heat.astype(float),
        annot=True,
        fmt=".0%",
        cmap="RdBu_r",
        vmin=0,
        vmax=1,
        linewidths=0.5,
        ax=ax,
        cbar_kws={"label": "AIO rate"},
    )
    ax.set_xlabel("Stance")
    ax.set_ylabel("Political leaning")
    ax.set_title("AIO presence rate — leaning × stance (test batch)")
    plt.tight_layout()
    plt.show()

    no_aio = df_lean[~df_lean["has_ai_overview"]][
        ["query", "topic", "subtopic", "pro_leaning", "stance"]
    ].sort_values(["pro_leaning", "stance", "topic"])
    print(
        f"\nQueries with no AIO: {len(no_aio)} / {len(df_lean)} ({len(no_aio) / len(df_lean):.1%})"
    )
    display(no_aio.reset_index(drop=True))

## 3. Consistency — repeated queries
For every query run more than once, did AIO appear consistently?

In [ ]:
consistency = (
    df.groupby("query")["has_ai_overview"]
    .agg(runs="count", aio_count="sum")
    .query("runs > 1")
    .assign(
        aio_rate=lambda x: x["aio_count"] / x["runs"],
        verdict=lambda x: x.apply(
            lambda r: (
                "always"
                if r["aio_count"] == r["runs"]
                else ("never" if r["aio_count"] == 0 else "inconsistent")
            ),
            axis=1,
        ),
    )
    .sort_values(["verdict", "query"])
)

if consistency.empty:
    print("No repeated queries in this test batch.")
else:
    print(consistency["verdict"].value_counts().to_string())
    display(consistency)

## 4. Query length (words)

In [ ]:
word_counts = df["query"].drop_duplicates().str.split().str.len()

print(f"Average query length : {word_counts.mean():.2f} words")
print(f"Median query length  : {word_counts.median():.0f} words")
print(word_counts.describe().to_string())